# Gece üretimi · öğretmen model
**Kim:** O dönemin Yapay Zekâ ekibi, kendi Colab hesabıyla.
**Nasıl:** Hücreleri çalıştır, sekmeyi açık bırak. Colab koparsa defteri yeniden çalıştır: **kaldığı yerden devam eder.**
İki hesap aynı anda çalışırsa `HESAP` farklı olsun, dosyalar karışmaz.

## 1 · Hazırlık
*Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU*** seçili olmalı.

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU YOK: T4 seç")

In [ ]:
# Drive: üretilen dosyalar buraya yazılır, Colab kopsa da kaybolmaz
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ileri-programlama/02_Veri'
!mkdir -p "$DRIVE/uretilen" "$DRIVE/raporlar"

In [ ]:
# Özel depolar için salt okunur GitHub token'ı; URL'ye/dosyaya kaydedilmez.
# Colab çıktılarında veya paylaşılacak notebook hücrelerinde token yazmayın.
import base64
import getpass
import os
from pathlib import Path
import subprocess

token = getpass.getpass('GitHub token (özel AI ve Docs depolarına okuma erişimi): ')
git_env = os.environ.copy()
if token:
    yetki = base64.b64encode(('x-access-token:' + token).encode()).decode()
    git_env.update(GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='Authorization: Basic ' + yetki)
    del yetki
git_env['GIT_TERMINAL_PROMPT'] = '0'
del token
try:
    for ad in ('AI', 'Docs'):
        adres = f'https://github.com/IleriProgramlamaProje/{ad}.git'
        hedef = Path('/content') / ad
        if not hedef.exists():
            subprocess.run(['git', 'clone', '--quiet', adres, str(hedef)], env=git_env, check=True)
        else:
            remote = subprocess.check_output(['git', '-C', str(hedef), 'remote', 'get-url', 'origin'], text=True).strip()
            if remote != adres:
                raise ValueError(f'{hedef}: beklenmeyen depo; klasörü kontrol edin')
            print(f'{ad}: mevcut kopya korunuyor; güncelleme gerekiyorsa önce yerel değişiklikleri kontrol edin.')
        print(ad, subprocess.check_output(['git', '-C', str(hedef), 'rev-parse', 'HEAD'], text=True).strip())
finally:
    git_env.clear()
os.chdir('/content/AI')
os.environ['PROJE_DOCS_DIR'] = '/content/Docs'


In [ ]:
# vLLM: toplu üretimde en hızlı motor (kurulum 3-5 dk). Kurulmazsa kod otomatik transformers'a düşer.
!pip install -q -U vllm transformers accelerate

In [ ]:
# Hugging Face girişi. Önce huggingface.co/google/gemma-4-E4B-it sayfasında lisansı kabul et.
from huggingface_hub import login
login()

## 2 · Üretim ayarı

In [ ]:
HESAP = 'a'          # ikinci hesapta 'b'
GECE  = '01'         # her gece bir artır: 01, 02, 03...
HIKAYE_HEDEF = 20000 # hız ölçümüne göre ayarla
KART_HEDEF   = 5000  # Ham üretim hedefi; eğitim karışım oranları token bazında ayrıca ayarlanır
import sys; sys.path.insert(0, '/content/AI')
from ai.veri_uretimi.uretici import uret

In [ ]:
uret('hikaye', HIKAYE_HEDEF, f'{DRIVE}/uretilen/hikaye_{HESAP}_{GECE}.jsonl', grup=8)

In [ ]:
uret('kart', KART_HEDEF, f'{DRIVE}/uretilen/kart_{HESAP}_{GECE}.jsonl', grup=8)

## 3 · Bitince
- Ertesi gün sayıları Discord'a yaz (kaç hikâye, kaç kart).
- *Çalışma zamanı → Bağlantıyı kes ve sil* (birim harcamasın).
- Filtre (`ai/filtre/`) 1. haftada yazılınca bu dosyalar ondan geçirilir.